# Hillstrom: CUPED, hồi quy và CUPAC, giảm được bao nhiêu variance?

Thí nghiệm email Hillstrom (module 03): 64 000 khách chia ngẫu nhiên vào `No E-Mail`, `Mens E-Mail`,
`Womens E-Mail`. Cột `history` là số tiền khách đã chi **trong 12 tháng trước** thí nghiệm, đúng loại
dữ liệu CUPED cần. Các cột hồ sơ khác (`recency`, `mens`, `womens`, `newbie`, `channel`, `zip_code`)
cũng có trước thí nghiệm.

Câu hỏi của notebook: với từng cách giảm variance, khoảng tin cậy hẹp lại bao nhiêu, và vì sao
con số thật khác xa lời hứa "CUPED giảm 50% variance" trong các bài blog.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (dòng đầu `# explore`) đọc dữ liệu thật, gọi hàm để xem. Test bỏ qua chúng.
- Chấm: `uv run pytest tracks/data-science/modules/04-advanced-experimentation/exercises/intermediate/01-hillstrom-cuped-cupac -q`

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import KFold

In [ ]:
# explore: đọc dữ liệu thật (tải bằng: uv run python data/download_dataset.py hillstrom)
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
df = pd.read_csv(ROOT / "data/raw/hillstrom/hillstrom.csv")
FEATURES = ["recency", "history", "mens", "womens", "newbie", "channel", "zip_code"]
print(df.shape)
df.head()

## 1. Covariate trước thí nghiệm mạnh cỡ nào?

CUPED giảm variance đúng bằng $$\rho^2$$, bình phương tương quan giữa metric và covariate. Trước khi
viết dòng code CUPED nào, hãy nhìn tương quan.

In [ ]:
# explore: tương quan của các cột trước thí nghiệm với từng metric
num = ["history", "recency", "mens", "womens", "newbie"]
corr = df[num + ["visit", "conversion", "spend"]].corr().loc[num, ["visit", "conversion", "spend"]]
print(corr.round(3))
print("rho^2 lớn nhất:", (corr**2).max().round(4).to_dict())

## 2. CUPED

`cuped_effect(df, metric, covariate, arm, control="No E-Mail") -> dict` với các khoá `diff`, `se`
(sau CUPED), `diff_raw`, `se_raw` (không điều chỉnh), `theta`, `var_reduction`.

- Chỉ dùng hai nhánh `arm` và `control`; `theta` tính trên dữ liệu gộp của hai nhánh đó.
- SE theo Welch. `var_reduction = 1 − var(y_cuped) / var(y)` trên dữ liệu hai nhánh.

In [ ]:
def cuped_effect(df: pd.DataFrame, metric: str, covariate: str, arm: str, control: str = "No E-Mail") -> dict:
    """Trả dict diff, se, diff_raw, se_raw, theta, var_reduction (chỉ dùng hai nhánh arm và control)."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: CUPED với history cho mọi nhánh × metric
rows = []
for arm in ["Mens E-Mail", "Womens E-Mail"]:
    for metric in ["visit", "conversion", "spend"]:
        out = cuped_effect(df, metric, "history", arm)
        rows.append({"arm": arm, "metric": metric, **out, "se_ratio": out["se"] / out["se_raw"]})
pd.DataFrame(rows).round(5)

## 3. Hồi quy nhiều covariate (regression adjustment)

CUPED một covariate tương đương hồi quy `metric ~ treat + covariate`. Hồi quy cho phép thêm **nhiều**
covariate, kể cả biến phân loại.

`regression_effect(df, metric, covariates, arm, control="No E-Mail") -> (diff, se)`: hệ số của biến
treatment trong OLS trên hai nhánh, SE robust `HC1`. Cột kiểu chuỗi đưa vào công thức dưới dạng
`C(cột)`. `covariates` rỗng thì là hiệu trung bình thông thường.

In [ ]:
def regression_effect(df: pd.DataFrame, metric: str, covariates: list[str], arm: str,
                      control: str = "No E-Mail") -> tuple[float, float]:
    """OLS metric ~ treat + covariates trên hai nhánh, SE HC1. Trả (hệ số treat, SE)."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: thêm dần covariate
for covs in [[], ["history"], ["history", "recency"], FEATURES]:
    diff, se = regression_effect(df, "visit", covs, "Womens E-Mail")
    print(f"{str(covs):75s} diff={diff:.5f} se={se:.6f}")

## 4. CUPAC: dùng dự đoán của model làm covariate

DoorDash gọi cách này là **CUPAC** (Control Using Predictions As Covariates): huấn luyện một model
dự đoán metric từ các đặc trưng có **trước** thí nghiệm, rồi dùng dự đoán làm covariate cho CUPED.
Model bắt được quan hệ phi tuyến và tương tác mà hồi quy tuyến tính bỏ sót.

Ở đây không có dữ liệu "kỳ trước" để huấn luyện, nên dùng **cross-fitting**: chia K fold, dự đoán
mỗi fold bằng model huấn luyện trên các fold còn lại. Dự đoán của một khách không bao giờ thấy
outcome của chính khách đó.

`cupac_covariate(df, features, metric, n_splits=5, seed=0) -> np.ndarray`: dự đoán out-of-fold cho
mọi dòng. Tuyệt đối không đưa `segment` vào features.

In [ ]:
def cupac_covariate(df: pd.DataFrame, features: list[str], metric: str, n_splits: int = 5,
                    seed: int = 0) -> np.ndarray:
    """Dự đoán out-of-fold của metric từ features (K-fold, xáo trộn với random_state=seed)."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: CUPAC cho visit và spend, so với CUPED một covariate
summary = []
for metric in ["visit", "spend"]:
    df[f"cupac_{metric}"] = cupac_covariate(df, FEATURES, metric)
    for arm in ["Mens E-Mail", "Womens E-Mail"]:
        base = cuped_effect(df, metric, "history", arm)
        cupac = cuped_effect(df, metric, f"cupac_{metric}", arm)
        summary.append({"metric": metric, "arm": arm, "diff_raw": base["diff_raw"],
                        "vr_history": base["var_reduction"], "vr_cupac": cupac["var_reduction"],
                        "diff_cupac": cupac["diff"], "se_raw": base["se_raw"], "se_cupac": cupac["se"]})
summary = pd.DataFrame(summary)
summary.round(5)

In [ ]:
# explore: quy ra traffic. Giảm variance v tương đương có thêm 1 / (1 − v) − 1 traffic.
summary["extra_traffic_cupac"] = 1 / (1 - summary.vr_cupac) - 1
summary[["metric", "arm", "vr_history", "vr_cupac", "extra_traffic_cupac"]].round(4)


## 5. Kết luận

TODO: 4–6 câu. Mỗi cách giảm được bao nhiêu variance trên Hillstrom? Vì sao ít như vậy? Với
`spend`, có đáng làm CUPAC không? Nếu là DS của nhà bán lẻ này, bạn sẽ thu thập covariate gì cho
thí nghiệm sau?
